# 01. 상권·주택·단속 좌표 전처리

입력: `data/raw/`의 공식 음식점·주택·단속 좌표와 행정동 경계.

출력: `data/processed/`의 좌표·출처 기록, `results/tables/`의 제외 목록.

기본 음식점 자료는 2022년입니다. 아래 DATASET을 official_2025로 바꾸면 2025년 자료를 사용합니다.

각 노트북은 별도 커널에서 실행할 수 있습니다. 공유 변수는 사용하지 않고 파일로 입력을 읽습니다. ZIP에는 기존 실행 결과도 들어 있습니다.

In [1]:
from pathlib import Path
import sys
# Supports Jupyter launched from the repository root or notebooks/.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('저장소 폴더 또는 notebooks 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.paths import ROOT, data_path, result_path

# 필요 시 주석 해제
# %pip install -r requirements.txt
from IPython.display import Image, HTML, display


In [2]:
"""좌표가 있는 공식 음식점 자료를 정리하고 과거 음식점 목록의 좌표도 보완한다."""
from src.paths import ROOT, data_path, result_path
from pathlib import Path
import argparse, hashlib, json, re
import numpy as np
import pandas as pd
from shapely.geometry import shape, Point
DEFAULT_DATASET='historical_2022'
SOURCE_PAGE='https://www.data.go.kr/data/15076735/fileData.do'
SOURCES={
 'historical_2022':('restaurant_basic_20220401.csv','2022-04-01','FILE_000000002865837'),
 'official_2025':('restaurant_basic_20250604.csv','2025-06-04','FILE_000000003152097')}

### `address_key`

In [3]:
def address_key(value,kind):
    s=re.sub(r'\s+',' ',str(value)).strip()
    if kind=='road':
        m=re.search(r'(완산구|덕진구)\s+([^\s,]+(?:로|길))\s+(\d+(?:-\d+)?)',s)
    else:
        m=re.search(r'(완산구|덕진구)\s+([^\s,]+(?:동\d*가|동))\s+(산\s*)?(\d+(?:-\d+)?)',s)
    if not m:return None
    if kind=='road':return '|'.join(m.groups())
    gu,dong,mountain,number=m.groups()
    return '|'.join([gu,dong,'산' if mountain else '',number])

### `read`

In [4]:
def read(path):
 for enc in ('utf-8-sig','cp949'):
  try:return pd.read_csv(path,encoding=enc,low_memory=False)
  except UnicodeDecodeError:continue
 raise ValueError(f'CSV 인코딩 확인 필요: {path}')

### `boundaries`

In [5]:
def boundaries():
 geo=json.loads((data_path('hangjeongdong_전라북도.geojson')).read_text())
 return [(f['properties']['adm_nm'].split()[-1],shape(f['geometry'])) for f in geo['features'] if '전주시' in f['properties']['adm_nm']]

### `assign_dong`

In [6]:
def assign_dong(d):
 polys=boundaries();results=[]
 for x,y in zip(d.Longitude,d.Latitude):
  if not np.isfinite(x) or not np.isfinite(y):results.append(None);continue
  hits=[dong for dong,poly in polys if poly.covers(Point(x,y))]
  results.append(hits[0] if len(hits)==1 else None)
 d=d.copy();d['행정동']=results
 return d

### `normalize_direct`

In [7]:
def normalize_direct(kind):
 filename,date,attachment=SOURCES[kind];path=data_path(filename)
 original=read(path)
 d=original.rename(columns={'식당명':'업소명','식당경도':'Longitude','식당위도':'Latitude'})
 for c in ('Longitude','Latitude'):d[c]=pd.to_numeric(d[c],errors='coerce')
 # 폐업/미확인은 기본 분석에서 제외, 신규/운영중은 포함
 d['eligible_status']=d['식당상태'].isin(['운영중','신규'])
 d['coordinate_valid']=d.Longitude.between(126,129)&d.Latitude.between(34,37)
 d=assign_dong(d)
 valid=d.eligible_status & d.coordinate_valid & d['행정동'].notna()
 included=d[valid].drop_duplicates(['업소명','도로명주소','지번주소']).copy()
 included['coordinate_status']='official_coordinates'
 included['coordinate_source']=SOURCE_PAGE+'; '+date
 included['source_date']=date
 excluded=d[~valid].copy()
 excluded.to_csv(result_path(f'commercial_excluded_{kind}.csv'),index=False,encoding='utf-8-sig')
 included.to_csv(data_path(f'commercial_points_{kind}.csv'),index=False,encoding='utf-8-sig')
 info={'dataset':kind,'source_url':SOURCE_PAGE,'source_date':date,'attachment_id':attachment,
       'download_url':f'https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId={attachment}&fileDetailSn=1&insertDataPrcus=N',
       'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'original_rows':len(original),
       'eligible_rows_before_coordinates':int(d.eligible_status.sum()),'excluded_rows':len(excluded),
       'unique_korean_restaurants':int(d.eligible_status.sum()),'coordinate_matched_restaurants':len(included),
       'unmatched_restaurants':int((d.eligible_status&~valid).sum()),
       'songcheon1_matched_restaurants':int((included['행정동']=='송천1동').sum()),
       'coordinate_method':'음식점 공식 원본의 위도/경도 직접 사용; 기존 GeoJSON 경계로 행정동 분류',
       'limitations':'원본 수록 음식점의 해당 기준일 분석; 전체 업소 완전성 또는 당시 발표 데이터와 동일함은 보장할 수 없음'}
 return included,info,d

### `enrich_original_catalog`

In [8]:
def enrich_original_catalog(direct_frames):
 old=read(data_path('restaurant_catalog.csv'))
 for c in ['Longitude','Latitude','행정동']:old[c]=None
 # 건물 좌표로만 연결하며 가게명이 다른 경우에도 동일 건물 주소는 좌표 참조로 사용 가능.
 refs={};conflicts=set()
 for frame in direct_frames:
  for kind,col in [('road','도로명주소'),('parcel','지번주소')]:
   for _,r in frame.iterrows():
    k=(kind,address_key(r[col],kind))
    if k[1] is None or not np.isfinite(r['Longitude']) or not np.isfinite(r['Latitude']):continue
    xy=(r['Longitude'],r['Latitude'])
    if k in refs:
     prev=refs[k];distance=np.hypot((prev[0]-xy[0])*90000,(prev[1]-xy[1])*111320)
     if distance>30:conflicts.add(k)
    else:refs[k]=xy
 rows=[]
 for _,r in old.iterrows():
  rec=r.to_dict();hits=[]
  for kind,col in [('road','도로명주소'),('parcel','지번주소')]:
   k=(kind,address_key(r[col],kind))
   if k in refs and k not in conflicts:hits.append((kind,refs[k]))
  rec.update(Longitude=np.nan,Latitude=np.nan,coordinate_status='unmatched_or_ambiguous')
  if hits:
   if len(hits)>1 and np.hypot((hits[0][1][0]-hits[1][1][0])*90000,(hits[0][1][1]-hits[1][1][1])*111320)>30:
    rec['coordinate_status']='conflicting_road_parcel'
   else:
    kind,(x,y)=hits[0];rec.update(Longitude=x,Latitude=y,coordinate_status=f'exact_official_building_{kind}')
  rows.append(rec)
 enriched=assign_dong(pd.DataFrame(rows))
 enriched.to_csv(data_path('restaurant_catalog_enriched.csv'),index=False,encoding='utf-8-sig')
 enriched[enriched['행정동'].isna()].to_csv(result_path('restaurants_without_coordinates.csv'),index=False,encoding='utf-8-sig')
 enriched[enriched['행정동'].notna()].to_csv(data_path('commercial_points_pdf_catalog.csv'),index=False,encoding='utf-8-sig')
 return {'catalog_rows':len(enriched),'coordinates_linked':int(enriched['행정동'].notna().sum()),
         'remaining_unmatched':int(enriched['행정동'].isna().sum()),
         'method':'2022/2025 official restaurant building addresses exact match; coordinates differing >30m rejected'}

### `prepare_point_layers`

In [9]:
def prepare_point_layers():
 for filename,name in [('shit_result.csv','housing_points'),('holy_result.csv','violation_points')]:
  d=read(data_path(filename))
  for c in ['Longitude','Latitude']:d[c]=pd.to_numeric(d[c],errors='coerce')
  d=assign_dong(d)
  d[d['행정동'].notna()].to_csv(data_path(f'{name}.csv'),index=False,encoding='utf-8-sig')
  d[d['행정동'].isna()].to_csv(result_path(f'{name}_excluded.csv'),index=False,encoding='utf-8-sig')

### `main`

In [10]:
def main(dataset=DEFAULT_DATASET):
 datasets={};raw=[];audit={}
 for kind in SOURCES:
  included,info,d=normalize_direct(kind);datasets[kind]=(included,info);raw.append(d);audit[kind]=info
 legacy=enrich_original_catalog(raw);prepare_point_layers()
 if dataset not in datasets:raise ValueError(f'dataset must be one of {list(datasets)}')
 included,info=datasets[dataset]
 included.to_csv(data_path('commercial_points.csv'),index=False,encoding='utf-8-sig')
 included.groupby('행정동').size().rename('음식점수').reset_index().to_csv(result_path('commercial_counts.csv'),index=False,encoding='utf-8-sig')
 info={**info,'all_sources':audit,'original_catalog_coordinate_enrichment':legacy}
 (data_path('commercial_provenance.json')).write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
 print(json.dumps({k:info[k] for k in ['dataset','coordinate_matched_restaurants','songcheon1_matched_restaurants','unmatched_restaurants']},ensure_ascii=False))
 return info

## 사용할 음식점 자료 선택 및 전처리

In [11]:
DATASET = 'historical_2022'
commercial_metadata = main(DATASET)
display(pd.read_csv(data_path('commercial_points.csv')).head())
display(pd.read_csv(result_path('commercial_counts.csv')))

,식당(ID),업소명,도로명주소,지번주소,Latitude,Longitude,식당대표전화번호,영업신고증업태명,영업인허가명,음식점소개내용,식당상태,eligible_status,coordinate_valid,행정동,coordinate_status,coordinate_source,source_date
0,1681,진미반점,전북특별자치도 전주시 완산구 전라감영3길 12-3,전북특별자치도 전주시 완산구 중앙동2가 44-1,35.816511,127.143362,063-284-4218,한식,일반음식점,"전라북도 전주시에 가신다면 ""진미반점""에 방문해보세요~ 지방자치단체 인증을 받은 농림축산식품부 제공 안심식당입니다. 50년 이상의 전통을 자랑합니다. 레드테이블에서 온라인 예약이 가능합니다! 주차공간, 다국어 메뉴판이 있는 매장입니다.",운영중,True,True,중앙동,official_coordinates,https://www.data.go.kr/data/15076735/fileData.do; 2022-04-01,2022-04-01
1,2691,한국식당,전북특별자치도 전주시 완산구 전라감영로 48-1,전북특별자치도 전주시 완산구 중앙동4가 34,35.815434,127.144495,063-284-6932,한식,일반음식점,"전라북도 전주시에 가신다면 ""한국식당""에 방문해보세요~ 지방자치단체 인증을 받은 농림축산식품부 제공 안심식당입니다. 레드테이블에서 온라인 예약이 가능합니다! 다국어 메뉴판이 있는 매장입니다.",운영중,True,True,중앙동,official_coordinates,https://www.data.go.kr/data/15076735/fileData.do; 2022-04-01,2022-04-01
2,3101,PNB풍년제과 본점,전북특별자치도 전주시 완산구 팔달로 180,전북특별자치도 전주시 완산구 경원동1가 40-5,35.818435,127.147196,063-285-6666,제과점영업,제과점영업,"2014년 09월 20일 Ystar 식신로드 200회, 2014년 05월 12일 SBS 생활의달인 429회, 2013년 09월 20일 KBS2 VJ특공대 683회 등 여러 방송에서 다룬 바로 그 곳! 전라북도 전주시 인기 맛집 ""피엔비풍년제과(경원동본점)""를 방문해보세요! 레드테이블에서 온라인 예약이 가능합니다! 배달 가능한 매장입니다. 다국어 메뉴판이 있는 매장입니다.",운영중,True,True,풍남동,official_coordinates,https://www.data.go.kr/data/15076735/fileData.do; 2022-04-01,2022-04-01
3,3103,가람나주곰탕,전북특별자치도 전주시 덕진구 떡전로 5,전북특별자치도 전주시 덕진구 금암동 794-7,35.835715,127.128254,063-227-3395,한식,일반음식점,"어디 가야 할지 고민이시라고요? 그럼 전라북도 전주시 맛집, ""가람 나주곰탕""을 추천합니다! 레드테이블에서 온라인 예약이 가능합니다! 배달 가능한 매장입니다. 주차공간, 다국어 메뉴판이 있는 매장입니다.",운영중,True,True,금암1동,official_coordinates,https://www.data.go.kr/data/15076735/fileData.do; 2022-04-01,2022-04-01
4,4121,태봉집,전북특별자치도 전주시 완산구 전주객사5길 43-14,전북특별자치도 전주시 완산구 고사동 195-1,35.820261,127.144530,063-283-2458,복어취급,일반음식점,"2019년 09월 27일 TV조선 식객허영만의백반기행 18회에서 소개되었던 맛집, ""태봉집""입니다! 전라북도 전주시 근처라면 방문해보세요. 레드테이블에서 온라인 예약이 가능합니다! 다국어 메뉴판이 있는 매장입니다.",운영중,True,True,중앙동,official_coordinates,https://www.data.go.kr/data/15076735/fileData.do; 2022-04-01,2022-04-01


,행정동,음식점수
0,금암1동,84
1,금암2동,18
2,노송동,32
3,덕진동,245
4,동서학동,20
5,삼천1동,8
6,삼천2동,41
7,삼천3동,85
8,서서학동,5
9,서신동,160


{"dataset": "historical_2022", "coordinate_matched_restaurants": 3144, "songcheon1_matched_restaurants": 260, "unmatched_restaurants": 5}
